# ML-08 · Capstone Modeling Lane
## 1. Method choice and why
**Refresh / Content Opportunity Scoring.** With a fixed editorial budget, rank pages for a refresh review using their estimated probability of losing more than 20% of impressions in the following 30-day window. Review is a decision-support action. A decline label does not measure whether editing would reverse it.

Start with regularized logistic regression. It gives a simple ranking, handles three log-transformed numeric features, and can be compared to a fixed rule without tuning on test clients. The operating pool is exactly the Week 4 queue: eligible pages with at least 500 prior impressions. Pages below that threshold stay outside both rankings.

This is the bundled 30,000-page snapshot. The held-out-client test checks transfer across clients within this snapshot. Testing later dates requires a separate warehouse panel evaluation.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

relative_input = Path("data/raw/content_refresh_anonymized.csv")
here = Path.cwd()
root = next((p for p in (here, *here.parents) if (p / relative_input).is_file()), None)
if root is None:
    from importlib.util import find_spec
    from hashlib import sha256
    from urllib.request import urlopen

    if find_spec("google") is None or find_spec("google.colab") is None:
        raise FileNotFoundError(f"Run inside the starter repository containing {relative_input}")

    root = here
    data_url = (
        "https://raw.githubusercontent.com/mhv-codes/flyrank-ml-internship/"
        "48d0dac47169e9f14b106037938d414b16529aaf/"
        f"{relative_input}"
    )
    with urlopen(data_url, timeout=60) as response:
        starter_csv = response.read()
    expected_sha256 = "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396"
    if sha256(starter_csv).hexdigest() != expected_sha256:
        raise ValueError("Downloaded starter CSV does not match the verified repository copy.")
    input_path = root / relative_input
    input_path.parent.mkdir(parents=True, exist_ok=True)
    input_path.write_bytes(starter_csv)

import numpy as np
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

keys = ["client_id", "content_id"]
raw_columns = keys + ["content_age_days", "impressions_prev_30d", "clicks_prev_30d", "impressions_last_30d"]
data = pd.read_csv(root / relative_input, usecols=raw_columns)
assert len(data) == 30000 and not data.duplicated(keys).any()
data["age_at_cutoff"] = data["content_age_days"] - 30
pool = data.loc[data.age_at_cutoff.ge(30) & data.impressions_prev_30d.ge(500)].copy()
features = ["impressions_prev_30d", "clicks_prev_30d", "age_at_cutoff"]
assert pool[keys + features + ["impressions_last_30d"]].notna().all().all()
assert pool[features].ge(0).all().all()
pool["decline"] = (5 * pool.impressions_last_30d < 4 * pool.impressions_prev_30d).astype(int)
print(f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}; queue candidates: {len(pool):,}")


pandas 2.2.3, scikit-learn 1.8.0; queue candidates: 11,119


## 2. Split design
Use a seeded 75/25 split by client. All pages for a client stay together. The model sees only impressions and clicks from the previous window and creation age rolled back 30 days. Latest-window impressions construct the outcome only. Client and content identifiers are used for split integrity and stable ranking ties. Both scores rank the same pages on the same test clients. K=50 is the primary review budget, with K=10 as a secondary check.

In [2]:
seed = 42
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed)
                           .split(pool, groups=pool.client_id))
train, test = pool.iloc[train_idx].copy(), pool.iloc[test_idx].copy()
assert set(train.client_id).isdisjoint(test.client_id)
assert train.decline.nunique() == test.decline.nunique() == 2
assert len(test) >= 50
print(pd.DataFrame([{"split": name, "pages": len(frame), "clients": frame.client_id.nunique(),
                     "decline_rate": frame.decline.mean()}
                    for name, frame in [("train", train), ("test", test)]]).round(4).to_string(index=False))


split  pages  clients  decline_rate
train  10336       21        0.5852
 test    783        7        0.9106


## 3. Train and compare with Week 4
Apply log1p to the nonnegative inputs and fit the scaler and logistic model on training pages. Keep the default regularization fixed. The baseline score is prior impressions. Sort both rankings by descending score and then ascending client and content keys, exactly matching the baseline tie policy. Precision@K is the share of pages with observed decline in the first K review slots. Average precision and ROC AUC summarize the full queue.

In [3]:
model = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                      LogisticRegression(max_iter=1000, random_state=seed))
model.fit(train[features], train.decline)
test["week4_score"] = test.impressions_prev_30d
test["logistic_score"] = model.predict_proba(test[features])[:, 1]
def rank(frame, score):
    return frame.sort_values([score, *keys], ascending=[False, True, True])
def metrics(name, score):
    ranked = rank(test, score)
    return {"method": name, "P@10": ranked.head(10).decline.mean(),
            "P@50": ranked.head(50).decline.mean(),
            "average precision": average_precision_score(test.decline, test[score]),
            "ROC AUC": roc_auc_score(test.decline, test[score])}
comparison = pd.DataFrame([metrics("Week 4 exposure rule", "week4_score"),
                           metrics("Logistic regression", "logistic_score")])


## 4. Errors and interpretation
Coefficients are associations for standardized log inputs. Prior clicks and impressions can be correlated, so interpret signs jointly. The age feature measures time since creation, so it cannot identify when a page was last refreshed. The case table uses anonymous case labels rather than publishing page identifiers.

In [4]:
ranked = rank(test, "logistic_score").copy()
ranked["review_rank"] = np.arange(1, len(ranked) + 1)
false_picks = ranked.head(50).query("decline == 0").head(2)
missed = ranked.iloc[50:].query("decline == 1").head(3 - len(false_picks))
cases = pd.concat([false_picks, missed]).head(3)
assert len(cases) == 3
print("Coefficients per standardized log unit:")
print(pd.Series(model[-1].coef_[0], index=features).round(4).to_string())
for i, row in enumerate(cases.itertuples(), 1):
    change = 100 * (row.impressions_last_30d / row.impressions_prev_30d - 1)
    if row.review_rank <= 50:
        reason = "High predicted risk used a slot, but observed change stayed above the decline cutoff."
    else:
        reason = "Observed decline qualified, but this candidate fell below the 50-slot budget."
    print(f"Case {i}: rank {row.review_rank}, score {row.logistic_score:.3f}, "
          f"prior impressions {row.impressions_prev_30d}, prior clicks {row.clicks_prev_30d}, "
          f"age {row.age_at_cutoff} days, actual change {change:+.1f}%. {reason}")
print("The snapshot cannot distinguish seasonality, demand changes, or changes to the page. Inspect those before recommending an edit.")


Coefficients per standardized log unit:
impressions_prev_30d    0.2864
clicks_prev_30d        -0.7160
age_at_cutoff          -0.3187
Case 1: rank 43, score 0.799, prior impressions 528, prior clicks 0, age 72 days, actual change -15.5%. High predicted risk used a slot, but observed change stayed above the decline cutoff.
Case 2: rank 51, score 0.789, prior impressions 1014, prior clicks 0, age 117 days, actual change -75.0%. Observed decline qualified, but this candidate fell below the 50-slot budget.
Case 3: rank 52, score 0.787, prior impressions 4171, prior clicks 1, age 117 days, actual change -78.9%. Observed decline qualified, but this candidate fell below the 50-slot budget.
The snapshot cannot distinguish seasonality, demand changes, or changes to the page. Inspect those before recommending an edit.


## 5. Self-check and final comparison
The baseline and model use identical test rows, label, review budgets and tie ordering. The baseline was reconstructed from the completed local Week 4 notebook. Compare the two review budgets before making a recommendation. For the next audit, revisit the FlyRank report’s Method & Limitations and ML appendix to distinguish its descriptive comparisons from predictive validation. [FlyRank research report](https://state-of-seo-2026.flyrank.ai/)

The test set has only 783 candidates across seven clients and a 91.1% decline base rate versus 58.5% in training. Candidate counts are uneven across clients, so a 25% client holdout is not a 25% page holdout. The 6-point gain at 50 slots is three additional observed declines and is specific to this split. A high queue precision is easier in this test set because decline is common. Before deployment, compare across multiple client splits and on an untouched future warehouse window.

In [5]:
assert test.week4_score.ge(500).all()
assert rank(test, "week4_score").index.is_unique
assert np.isfinite(test.logistic_score).all()
print(f"Held-out decline base rate: {test.decline.mean():.3f}")
print(comparison.round(4).to_string(index=False))
for k in (10, 50):
    delta = comparison.loc[1, f"P@{k}"] - comparison.loc[0, f"P@{k}"]
    print(f"P@{k} model minus baseline: {delta * 100:+.1f} percentage points")
if comparison.loc[1, "P@50"] <= comparison.loc[0, "P@50"]:
    print("Keep the Week 4 baseline for the 50-slot review budget. This run did not establish a gain.")
else:
    print("The model improved the 50-slot queue on these held-out clients. Verify on a later window before adopting it.")


Held-out decline base rate: 0.911
              method  P@10  P@50  average precision  ROC AUC
Week 4 exposure rule   0.8  0.92             0.9309   0.6205
 Logistic regression   1.0  0.98             0.9483   0.6463
P@10 model minus baseline: +20.0 percentage points
P@50 model minus baseline: +6.0 percentage points
The model improved the 50-slot queue on these held-out clients. Verify on a later window before adopting it.
